# Lab 3.5 &mdash; Challenge &mdash; Shared State and Context Poisoning

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 1 &middot; Module 3 &mdash; Memory, State &amp; the LangGraph Substrate**

### What you'll do
- Run three agents over one shared state and watch a wrong finding spread
- Attach provenance so a claim can be checked instead of the consensus
- Build a critic that verifies against the source, not against agreement
- Compare shared and private state on cost, containment and auditability

> **How this lab works.** Fill every `BLANK`, then run the **Self-check** cell under each section.
> Graded cells are plain Python and never call a model, so your score never depends on a
> live endpoint. Cells marked **Run it for real** do call the sandbox model; if it is not
> reachable they print how to fix it instead of crashing.

> **The comprehensive lab for Module 3, and the bridge into Day 2.** Everything so far
> has been one agent. This is what memory does when there are several.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-3-05")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These two values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- graded cells still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 1 labs: payment exceptions on a small ledger.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

In [ ]:
# ------------------------------------------------- carried forward from Lab 1.2 of Module 1
# These are the tools you wrote in Lab 1.2 of Module 1. Nothing to fill in -- they are here so this
# notebook runs on its own. Note the docstrings: they name the case AND the boundary.

def lookup_payment(ref: str) -> str:
    """Return the ledger record for one payment reference such as 'PMT-1002'.

    Use when you need the status, amount, counterparty or reason code of a specific payment.
    Not for searching across payments.
    """
    record = LEDGER.get(ref)
    if record is None:
        return f"no payment found with reference {ref!r}"
    return json.dumps({"ref": ref, **record})


def policy_for(reason_code: str) -> str:
    """Return the operating policy for one failure reason code, e.g. 'LIMIT_BREACH'.

    Use after you know why a payment failed and need to know what to do about it.
    """
    return POLICY.get(reason_code, f"no policy on file for reason code {reason_code!r}")


TOOLS = {"lookup_payment": lookup_payment, "policy_for": policy_for}
print("carried forward:", ", ".join(TOOLS))

## Concept

Shared state is cheap and consistent. It is also how one agent's error becomes every agent's
premise &mdash; and because each later agent reasons correctly *from what it was told*, the system ends
up confidently and unanimously wrong, with nothing erroring.

Three agents agreeing is not corroboration. It is an echo. The defence is **provenance**.

## Section 1 &mdash; A finding that can be checked

A bare string cannot be verified. A finding that carries its author and its source can.

In [ ]:
def finding(claim: str, *, by: str, source: str, ref: str) -> dict:
    """One finding, with enough provenance that a critic can re-derive it."""
    return {"claim": claim, "by": by, "source": source, "ref": ref}

def verifiable(f: dict) -> bool:
    """True when a finding carries everything needed to check it independently."""
    required = ("claim", "by", "source", "ref")
    if not all(f.get(k) for k in required):
        return False
    return BLANK                     # TODO: the source must be one a critic can actually re-read.
                                     # Accept only "ledger" or "policy".

In [ ]:
# --- Self-check: Section 1
_good = finding("PMT-1005 is held", by="ledger_agent", source="ledger", ref="PMT-1005")
_hearsay = finding("PMT-1005 is held", by="critic", source="another agent said so", ref="PMT-1005")

check("a sourced finding is verifiable", lambda: verifiable(_good) is True)
check("a finding sourced from another agent is not",
      lambda: verifiable(_hearsay) is False,
      "if the source is hearsay, checking it only re-checks the echo")
check("a finding missing its author is not verifiable",
      lambda: verifiable({**_good, "by": ""}) is False)
check("a finding missing its reference is not verifiable",
      lambda: verifiable({**_good, "ref": ""}) is False)

## Section 2 &mdash; Watch the poison spread

The ledger agent misreads one field. Everything downstream is correct given what it was told.

In [ ]:
def ledger_agent(state, *, faulty=False):
    """Reads the ledger. With faulty=True it misreads a held payment as failed."""
    rec = LEDGER.get(state["ref"], {})
    status = rec.get("status", "unknown")
    if faulty and status == "held":
        status = "failed"                              # the single wrong bit
    return {"findings": [finding(f"{state['ref']} is {status}",
                                 by="ledger_agent", source="ledger", ref=state["ref"])]}

def policy_agent(state):
    """Reads policy for whatever the ledger said. Correct, given its input."""
    said = state["findings"][0]["claim"]
    code_ = "INSUFFICIENT_FUNDS" if "failed" in said else LEDGER.get(state["ref"], {}).get("reason_code")
    return {"findings": [finding(f"policy: {policy_for(code_)}",
                                 by="policy_agent", source="policy", ref=state["ref"])]}

def naive_critic(state):
    """Checks that the findings agree with each other. This is the trap."""
    claims = [f["claim"] for f in state["findings"]]
    consistent = not ("held" in " ".join(claims) and "failed" in " ".join(claims))
    return {"findings": [finding(f"consistency check: {'consistent' if consistent else 'conflict'}",
                                 by="critic", source="other agents", ref=state["ref"])]}

def sourced_critic(state):
    """Re-derives each verifiable finding from its named source. This is the fix."""
    problems = []
    for f in state["findings"]:
        if not verifiable(f):
            continue
        if f["source"] == "ledger":
            truth = LEDGER.get(f["ref"], {}).get("status", "unknown")
            if BLANK:                # TODO: does the claim disagree with the ledger?
                problems.append(f"{f['by']} claimed '{f['claim']}' but the ledger says '{truth}'")
    return {"findings": [finding(f"source check: {problems or 'all findings match their sources'}",
                                 by="sourced_critic", source="ledger", ref=state["ref"])],
            "problems": problems}

In [ ]:
# --- Self-check: Section 2
def run_shared(ref="PMT-1005", faulty=False, critic=naive_critic):
    state = {"ref": ref, "findings": [], "problems": []}
    for node in (lambda s: ledger_agent(s, faulty=faulty), policy_agent, critic):
        upd = node(state)
        state = {**state, **{k: v for k, v in upd.items() if k != "findings"},
                 "findings": state["findings"] + upd["findings"]}
    return state

_clean = run_shared(faulty=False)
_poisoned = run_shared(faulty=True)

check("a clean run reads the payment as held",
      lambda: "held" in _clean["findings"][0]["claim"])
check("the faulty run reads it as failed", lambda: "failed" in _poisoned["findings"][0]["claim"])
check("the policy agent proceeds correctly from the WRONG premise",
      lambda: "Retry" in _poisoned["findings"][1]["claim"],
      "it applied the retry policy -- correct, given what it was told")
check("the naive critic sees no problem at all",
      lambda: "consistent" in _poisoned["findings"][2]["claim"],
      "it checked agreement, and the agents did agree")
check("the sourced critic catches it",
      lambda: len(run_shared(faulty=True, critic=sourced_critic)["problems"]) == 1,
      "re-derive the claim from the ledger rather than comparing it with other agents")
check("the sourced critic does not cry wolf on a clean run",
      lambda: run_shared(faulty=False, critic=sourced_critic)["problems"] == [])

for name, st in (("clean", _clean), ("poisoned", _poisoned)):
    try:
        print(f"--- {name} ---")
        for f in st["findings"]:
            print(f"   [{f['by']:16} src={f['source']:14}] {f['claim'][:70]}")
    except NameError:
        print("(fill in the blanks above)"); break

## Section 3 &mdash; Private state contains it

Give each agent its own state and an explicit handoff. The error stops travelling &mdash; and you pay
for that in re-sent context.

In [ ]:
def run_private(ref="PMT-1005", faulty=False):
    """Each agent gets only what the previous one explicitly handed over."""
    tokens = 0
    ledger_state = {"ref": ref, "findings": []}
    ledger_out = ledger_agent(ledger_state, faulty=faulty)
    tokens += len(json.dumps(ledger_out)) // 4

    # the handoff: the policy agent receives the CLAIM, and re-reads the ledger itself
    policy_state = {"ref": ref, "findings": ledger_out["findings"]}
    checked = sourced_critic(policy_state)
    tokens += len(json.dumps(policy_state)) // 4       # context re-sent at the boundary

    if checked["problems"]:
        return {"outcome": "handoff rejected", "problems": checked["problems"], "tokens": tokens}
    policy_out = policy_agent(policy_state)
    tokens += len(json.dumps(policy_out)) // 4
    return {"outcome": BLANK, "problems": [], "tokens": tokens}
                                     # TODO: what should a clean private run report?

In [ ]:
# --- Self-check: Section 3
check("a clean private run completes",
      lambda: run_private(faulty=False)["outcome"] == "completed")
check("a poisoned private run is rejected at the handoff",
      lambda: run_private(faulty=True)["outcome"] == "handoff rejected",
      "checking at the boundary is what containment means")
check("the rejection names the disagreement",
      lambda: "ledger says" in run_private(faulty=True)["problems"][0])
check("containment is not free",
      lambda: run_private(faulty=False)["tokens"] > 0,
      "context is re-sent at every boundary -- that is the coordination tax from Module 1")

## Section 4 &mdash; The comparison, on four axes

Cost, containment, auditability, and whether the wrong answer reached the end.

In [ ]:
def comparison() -> str:
    rows = [f"{'design':22}{'poison contained':>18}{'tokens':>9}{'audit granularity':>20}",
            "-" * 72]
    shared_naive = run_shared(faulty=True, critic=naive_critic)
    shared_sourced = run_shared(faulty=True, critic=sourced_critic)
    private = run_private(faulty=True)
    rows.append(f"{'shared + naive critic':22}{'no':>18}{'low':>9}{'one trace':>20}")
    rows.append(f"{'shared + sourced critic':22}{'yes':>18}{'low':>9}{'one trace':>20}")
    rows.append(f"{'private + handoff check':22}{'yes':>18}{private['tokens']:>9}{'per agent':>20}")
    rows.append("")
    rows.append(f"naive critic found {len(shared_naive.get('problems', []))} problems; "
                f"sourced critic found {len(shared_sourced['problems'])}.")
    return "\n".join(rows)

try:
    print(comparison())
except NameError:
    print("(finish the sections above, then re-run this cell)")

In [ ]:
# --- Self-check: Section 4
check("the comparison covers all three designs",
      lambda: len(comparison().splitlines()) == 7)
check("the naive critic is recorded as catching nothing",
      lambda: "found 0 problems" in comparison())
check("the sourced critic is recorded as catching it",
      lambda: "found 1" in comparison())
check("shared state with a sourced critic is enough to contain the error",
      lambda: len(run_shared(faulty=True, critic=sourced_critic)["problems"]) == 1,
      "you do not need private state -- you need a critic that checks sources")

## Run it for real

Give the model both sets of findings and ask which it trusts. Watch whether provenance changes its
answer &mdash; and notice that you are testing your *design*, not the model.

In [ ]:
if llm_ready():
    try:
        poisoned = run_shared(faulty=True, critic=sourced_critic)
        rendered = "\n".join(
            f"- [{f['by']}, source={f['source']}] {f['claim']}" for f in poisoned["findings"])
        verdict = ask(
            "These are the findings from three agents working one payment case. State whether the "
            "case is safe to action, and if any finding should be distrusted, say which and why. "
            "Judge each finding by its source, not by whether the others agree with it.\n\n"
            + rendered)
        print(rendered)
        print("\n--- verdict ---\n" + verdict.strip()[:500])
    except NameError:
        print("(finish the sections above, then re-run this cell)")

### Read it

If the model flags the ledger finding because the source check contradicts it, provenance did the
work &mdash; not the model's judgement. Remove the `source=` labels and re-run: the same model, given the
same claims without provenance, has nothing to reason from but agreement.

**What you take from Module 3:** memory that survives a long session, observations the model cannot
misread, state you can print and store, and the checkpoint history that answers an auditor.
Day 2 puts several agents on top of this &mdash; and now you know what that does to a shared memory.

In [ ]:
score()

## Your turn

1. `sourced_critic` only re-checks findings sourced from the ledger. Extend it to policy findings.
   What stops a critic from simply becoming a second, equally fallible agent?
2. The poisoned run had exactly one wrong bit. Make the ledger agent wrong *intermittently* &mdash; a
   third of the time &mdash; and decide how the design should respond to a source that is usually right.
3. Combine this with Lab 3.4: at which checkpoint would an auditor first have been able to see the
   contradiction? That answer is your detection latency, and it is a number worth knowing.